In [1]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

# Main project folder
base_path = "C:/Users/User/Desktop/imageProcessing"

# Member 4 frames folder
input_folder = os.path.join(base_path, "Frames")

# Your output folders
output_folder = os.path.join(base_path, "Contrast_Output")
comparison_folder = os.path.join(base_path, "Contrast_Comparisons")

# Create output folders if not exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(comparison_folder, exist_ok=True)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

PermissionError: [WinError 5] Access is denied: 'C:/Users/User'

In [ ]:
frame_files = sorted(
    [f for f in os.listdir(input_folder)
     if f.startswith("frame_") and f.endswith(".jpg")],
    key=lambda x: int(x.split("_")[1].split(".")[0])
)

In [ ]:
std_results = []
all_pixels_before = []
all_pixels_after = []

comparison_saved = 0
count = 0

for file in frame_files:

    img_path = os.path.join(input_folder, file)
    img = cv2.imread(img_path)

    if img is None:
        continue

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply CLAHE
    contrast_img = clahe.apply(gray)

    # Save enhanced image (ALL frames)
    save_path = os.path.join(output_folder, file)
    cv2.imwrite(save_path, contrast_img)

    # Standard deviation
    std_before = np.std(gray)
    std_after = np.std(contrast_img)
    std_results.append([file, std_before, std_after])

    # Collect pixels for overall histogram
    all_pixels_before.extend(gray.ravel())
    all_pixels_after.extend(contrast_img.ravel())

    # Save first 3 comparison images
    if comparison_saved < 3:
        combined = np.hstack((gray, contrast_img))
        comp_path = os.path.join(
            comparison_folder,
            f"comparison_{comparison_saved}.jpg"
        )
        cv2.imwrite(comp_path, combined)

        # Histogram before
        plt.figure()
        plt.hist(gray.ravel(), bins=256)
        plt.title("Histogram Before CLAHE")
        plt.xlabel("Pixel Intensity")
        plt.ylabel("Frequency")
        plt.savefig(os.path.join(
            comparison_folder,
            f"hist_before_{comparison_saved}.png"
        ))
        plt.close()

        # Histogram after
        plt.figure()
        plt.hist(contrast_img.ravel(), bins=256)
        plt.title("Histogram After CLAHE")
        plt.xlabel("Pixel Intensity")
        plt.ylabel("Frequency")
        plt.savefig(os.path.join(
            comparison_folder,
            f"hist_after_{comparison_saved}.png"
        ))
        plt.close()

        comparison_saved += 1

    count += 1

In [ ]:
print("\nStandard Deviation Values:\n")
print("Frame\t\tBefore\t\tAfter")

for row in std_results:
    print(f"{row[0]}\t{row[1]:.4f}\t{row[2]:.4f}")


In [ ]:
avg_before = np.mean([r[1] for r in std_results])
avg_after = np.mean([r[2] for r in std_results])

print("\nAverage Standard Deviation:")
print("Before CLAHE:", round(avg_before, 4))
print("After CLAHE:", round(avg_after, 4))
print("Average Improvement:", round(avg_after - avg_before, 4))

In [ ]:
plt.figure()
plt.hist(all_pixels_before, bins=256)
plt.title("Overall Histogram Before CLAHE")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.savefig(os.path.join(base_path, "overall_hist_before.png"))
plt.close()

plt.figure()
plt.hist(all_pixels_after, bins=256)
plt.title("Overall Histogram After CLAHE")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.savefig(os.path.join(base_path, "overall_hist_after.png"))
plt.close()

print("\nTotal frames processed:", count)
print("Enhanced images saved in Contrast_Output folder")
print("Comparison images saved in Contrast_Comparisons folder")
print("Overall histograms saved successfully")